# 🤖 AI Job Application Agent

A multi-agent system that runs your job search for you, every morning.

**How it works — three agents in a pipeline:**
1. **Search agent** — queries Google Jobs (via SerpAPI) for your target roles.
2. **Filter agent** — Gemini scores each posting 0–100 against your resume and flags spam.
3. **Email agent** — sends the top matches to your inbox.

**The trigger:** a scheduler fires the whole pipeline automatically at 8 AM daily — no one has to press run.

**Make it yours:** everything you'd change lives in one **Config** cell below — your target roles, location, email, and resume. Swap those four things and the agent works for anyone.

In [1]:
# Cell 1 — Install dependencies
!pip install -q google-generativeai serpapi google-search-results schedule

  Preparing metadata (setup.py) ... done


In [2]:
# Cell 2 — Imports and load secrets
import os
import json
import time
import smtplib
import google.generativeai as genai
from serpapi import GoogleSearch
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from kaggle_secrets import UserSecretsClient

# Load secrets (set these under Add-ons → Secrets in the Kaggle editor)
secrets = UserSecretsClient()
SERPAPI_KEY        = secrets.get_secret("SERPAPI_KEY")
GEMINI_API_KEY     = secrets.get_secret("GEMINI_API_KEY")
GMAIL_USER         = secrets.get_secret("GMAIL_USER").replace("\xa0", "").replace(" ", "").strip()
GMAIL_APP_PASSWORD = secrets.get_secret("GMAIL_APP_PASSWORD").replace("\xa0", "").replace(" ", "").strip()

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

print("✅ All secrets loaded and Gemini configured.")

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


✅ All secrets loaded and Gemini configured.


In [3]:
# Cell 3 — Search Agent (cleaned up)
def search_jobs(query, location="United States"):
    params = {
        "engine": "google_jobs",
        "q": query,
        "location": location,
        "api_key": SERPAPI_KEY,
    }
    results = GoogleSearch(params).get_dict()

    if results.get("error"):
        print(f"⚠️ SerpAPI error: {results['error']}")
        return []
    return results.get("jobs_results", [])

In [ ]:
# Cell 3 — CONFIG  (everything reusable lives here — change these to retarget the agent)

# --- What to search for ---
TARGET_ROLES = [
    "entry level Data Analyst remote OR hybrid USA OR Canada",
    "entry level Business Intelligence Analyst remote OR hybrid USA OR Canada",
]
LOCATION = "United States"          # a real place Google Jobs understands

# --- Where results go ---
RECIPIENT_EMAIL = GMAIL_USER        # who receives the daily digest

# --- When it runs ---
SCHEDULE_TIME = "08:00"
TIMEZONE      = "America/New_York"

# --- The resume the agent scores jobs against (swap in your own) ---
RESUME_SUMMARY = """
Name: Example Candidate
Education: BS Computer Science, State University

Skills:
- Programming: Python, SQL, JavaScript
- Data Tools: Pandas, PostgreSQL
- Other: Git/GitHub

Experience:
- Data Analyst Intern, Example Corp (2024): built reporting dashboards

Target roles: Data Analyst, Business Intelligence Analyst
Preferences: Entry-level, remote or hybrid, USA or Canada
"""

In [5]:
# Cell 4 — Filter Agent (robust: retries, safe JSON parsing, fails OPEN)
import re

def score_job(job, retries=3):
    title       = job.get("title", "")
    company     = job.get("company_name", "")
    location    = job.get("location", "")
    description = job.get("description", "")[:1500]

    prompt = f"""
You are a job fit evaluator. Score this job posting for the candidate below from 0-100.
Only mark is_spam=true for fake listings, content farms, or obvious scams — NOT for real companies.

CANDIDATE:
{RESUME_SUMMARY}

JOB POSTING:
Title: {title}
Company: {company}
Location: {location}
Description: {description}

Respond ONLY with JSON, no other text:
{{"score": <0-100>, "is_spam": <true/false>, "reason": "<one sentence>"}}
"""
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            text = response.text.strip()
            match = re.search(r"\{.*\}", text, re.DOTALL)   # pull the JSON object out
            if match:
                return json.loads(match.group())
            raise ValueError(f"No JSON found: {text[:120]}")
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(5)            # back off (likely a rate limit) and retry
                continue
            return {"score": 0, "is_spam": False, "reason": f"Eval failed: {e}"}

In [6]:
# Cell 5 — Email Agent (UTF-8 safe)
def send_email(jobs):
    if not jobs:
        print("No jobs to email.")
        return

    parts = ["<h2>Top job matches today</h2>"]
    for j in jobs:
        opts = j.get("apply_options") or []
        link = opts[0].get("link", "") if opts else ""
        apply_html = f'<a href="{link}">Apply</a>' if link else ""
        parts.append(f"<p><b>{j.get('title','')}</b> @ {j.get('company_name','')}<br>"
                      f"📍 {j.get('location','')} — <b>Score: {j.get('score','')}</b><br>"
                      f"{j.get('reason','')}<br>{apply_html}</p>")

    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"Job Agent: {len(jobs)} new matches"
    msg["From"]    = GMAIL_USER
    msg["To"]      = RECIPIENT_EMAIL
    msg.attach(MIMEText("\n".join(parts), "html", "utf-8"))   # ← "utf-8" added

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(GMAIL_USER, GMAIL_APP_PASSWORD)
        server.send_message(msg)                              # ← was sendmail(...)
    print(f"📧 Emailed {len(jobs)} matches to {RECIPIENT_EMAIL}")

In [7]:
# Cell 6 — Orchestrator
def run_pipeline():
    print("🚀 Pipeline started\n")

    # 1) SEARCH — loop over every role in the config
    all_jobs = []
    for role in TARGET_ROLES:
        print(f"🔍 Searching: {role}")
        all_jobs += search_jobs(role, LOCATION)
    print(f"✅ Found {len(all_jobs)} postings\n")

    # 2) SCORE + FILTER
    print("🤖 Gemini is scoring each job against your resume...\n")
    scored = []
    for job in all_jobs:
        result = score_job(job)
        if not result.get("is_spam", True):
            scored.append({**job, **result})
            print(f"✅ {job.get('title')} @ {job.get('company_name')} — Score: {result['score']}")
        else:
            print(f"🚫 Filtered ({result.get('reason','')}): {job.get('title')} @ {job.get('company_name')}")
        time.sleep(2)
    top_jobs = sorted(scored, key=lambda x: x["score"], reverse=True)[:10]
    print(f"\n🏆 {len(top_jobs)} top matches selected\n")

    # 3) EMAIL
    send_email(top_jobs)
    print("\n✅ Pipeline finished")
    return top_jobs

run_pipeline()

🚀 Pipeline started

🔍 Searching: entry level Data Analyst remote OR hybrid USA OR Canada
🔍 Searching: entry level Business Intelligence Analyst remote OR hybrid USA OR Canada
✅ Found 8 postings

🤖 Gemini is scoring each job against your resume...

✅ Entry-Level Data Analyst: Visualization & Data Storytelling @ MaziCTools — Score: 70
✅ REMOTE – Full Time Data Entry Analyst @ Rental Consulting — Score: 90
✅ Entry Level Business Data Analyst @ RemoteMQ — Score: 98
✅ Junior - Intermediate Business Intelligence Data Analyst @ BEOVEXIS — Score: 0
✅ Business Intelligence Analyst - Entry Level @ hirevista — Score: 0
✅ Business Intelligence Analyst - People Analytics @ Workfromhome — Score: 0
✅ Remote Business Intelligence Developer Job for Financial Company (Full-time) @ remotepromsp — Score: 0
✅ Business Intelligence Analyst, Workplace Experience @ Adobe — Score: 0

🏆 8 top matches selected

📧 Emailed 8 matches to you@example.com

✅ Pipeline finished


[{'title': 'Entry Level Business Data Analyst',
  'company_name': 'RemoteMQ',
  'location': 'Anywhere',
  'via': 'Remote Jobs',
  'share_link': 'https://www.google.com/search?ibp=htl;jobs&q=entry+level+Data+Analyst+remote+OR+hybrid+USA+OR+Canada&htidocid=iNBZVNmjQxLn_QdEAAAAAA%3D%3D&hl=en-US&shem=epsd1,rimspwouoe&shndl=37&shmd=H4sIAAAAAAAA_xXEsQrCQAwAUFy7ujllFrwTQRCdFEUQHdQPKGkJd5U0KU2U9h_8aPENr_hOis1JvB_hSh9iOLytETKDIzrCXpBHc1jARSswwr7OoAJn1cQ022X3zrYxmnFI5uhNHWptowpVOsSXVvavtIw9dYxO5Wq9HEInaT59UKtOtzs0Ak8SSsg_QNNcg48AAAA&shmds=v1_ARwrE20Jr86Z3aq6TJ4wiVrlBuRgttBQRH2MyHaJwCavVD8O_A&source=sh/x/job/li/m1/1#fpstate=tldetail&htivrt=jobs&htiq=entry+level+Data+Analyst+remote+OR+hybrid+USA+OR+Canada&htidocid=iNBZVNmjQxLn_QdEAAAAAA%3D%3D',
  'extensions': ['6 days ago', 'Work from home', 'Full-time'],
  'detected_extensions': {'posted_at': '6 days ago',
   'work_from_home': True,
   'schedule_type': 'Full-time'},
  'source_link': 'https://remote.mq.unaux.com/remote-jobs/entry-level-business-d

In [8]:
# Cell 7 — Scheduler / Trigger (the agent runs itself every morning at 8 AM)
import schedule

SCHEDULE_TIME = "08:00"
TIMEZONE      = "America/New_York"   # 8 AM your time, handles DST automatically

def daily_job():
    print(f"\n⏰ Trigger fired at {SCHEDULE_TIME} {TIMEZONE} — running one search round")
    run_pipeline()

# Register the trigger: one run every day at 8 AM
schedule.every().day.at(SCHEDULE_TIME, TIMEZONE).do(daily_job)
print(f"✅ Agent scheduled: one search round every day at {SCHEDULE_TIME} {TIMEZONE}")
print(f"   Next scheduled run: {schedule.next_run()}")

# DEMO_MODE  → run one round right now so the notebook shows real output (use this when submitting)
# LIVE_MODE  → keep the agent alive and let it fire by itself at 8 AM each day
DEMO_MODE = True

if DEMO_MODE:
    print("\n🧪 Demo: triggering one round now to prove the scheduler works...\n")
    daily_job()
else:
    print("\n🟢 Live: agent is now running. It will trigger automatically at 8 AM daily.")
    while True:
        schedule.run_pending()
        time.sleep(30)

✅ Agent scheduled: one search round every day at 08:00 America/New_York
   Next scheduled run: 2026-06-30 12:00:00

🧪 Demo: triggering one round now to prove the scheduler works...


⏰ Trigger fired at 08:00 America/New_York — running one search round
🚀 Pipeline started

🔍 Searching: entry level Data Analyst remote OR hybrid USA OR Canada
🔍 Searching: entry level Business Intelligence Analyst remote OR hybrid USA OR Canada
✅ Found 8 postings

🤖 Gemini is scoring each job against your resume...

✅ Entry-Level Data Analyst: Visualization & Data Storytelling @ MaziCTools — Score: 0
✅ REMOTE – Full Time Data Entry Analyst @ Rental Consulting — Score: 0
✅ Entry Level Business Data Analyst @ RemoteMQ — Score: 0
✅ Junior - Intermediate Business Intelligence Data Analyst @ BEOVEXIS — Score: 0
✅ Business Intelligence Analyst - Entry Level @ hirevista — Score: 0
✅ Business Intelligence Analyst - People Analytics @ Workfromhome — Score: 0
✅ Remote Business Intelligence Developer Job for Financial